# EEG Processing

In this notebook, we assess the quality of the EEG recordings. If possible, we process the EEG data to enhance its quality.

## 📦 Imports

In [1]:
%matplotlib qt

from pathlib import Path 
import os 

import mindmap as mm

import mne

## 🧠 Data

The raw EEG contains the scalp potentials recorded over the entire duration. If onset times are provided, the data can be divided into epochs, which can then be averaged to form evoked potentials. Evoked potentials have the advantage of a strong signal-to-noise ratio, as the noise is averaged out. However, they require onset times, which may not be available in certain situations, such as during resting state recordings.

In [2]:
sub     = "sub-pat02"
mm.io.read_config("../configurations/spike.yml")

# directories
mm_dir = Path(os.environ["MM_DIR"])

# data
data = mm.io.read_dataset(sub, "eeg_proc")

## Raw

#### <font color=yellow>🌟</font> Visualize Raw

In [3]:
_ = data["eeg_proc"].plot()

In [4]:
_ = data["eeg_proc"].compute_psd().plot()

    Using multitaper spectrum estimation with 7 DPSS windows
Plotting power spectral density (dB=True).


/tmp/ipykernel_113517/2658904441.py:1: UserWarning: Zero value in spectrum for channel Cz
  _ = data["eeg_proc"].compute_psd().plot()
/tmp/ipykernel_113517/2658904441.py:1: UserWarning: Infinite value in PSD for channel Cz.
These channels might be dead.
  _ = data["eeg_proc"].compute_psd().plot()


#### Process Raw

⚙️ <font color=green>**Interpolate bad channels**</font>: After observing the time series and Fourier transform, we can identify any electrodes that did not record properly. In such cases, we interpolate the faulty channels to prevent them from affecting the rest of the analysis.

In [6]:
data["eeg_proc"].info["bads"].append("7")

data["eeg_proc"].interpolate_bads()

Setting channel interpolation method to {'eeg': 'spline'}.


Interpolating bad channels.
    Automatic origin fit: head of radius 88.7 mm
Computing interpolation matrix from 203 sensor positions
Interpolating 1 sensors


<Evoked | 'No comment' (average, N=1), 0 – 0.996 s, baseline off, 204 ch, ~633 kB>

⚙️ <font color=green>**Set reference**</font>: We can also determine whether the reference electrode is too close to the neural sources. An ideal reference signal captures none of the brain-specific fluctuations in electric potential while picking up all of the environmental noise and interference detected by the measurement electrodes. In practice, this means that the reference electrode is often placed on the subject's body, close to the head but as far away from the neural sources as possible. If the reference electrode also records neural sources, we can set a "virtual reference," which is the average of all channels.

In [10]:
data["eeg_proc"].set_eeg_reference(ref_channels="average")
_ = data["eeg_proc"].plot()

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.


## Epoched

## Evoked

In [11]:
if os.environ["NAME"] == "spike":
    evoked = data["eeg_proc"].copy()

elif os.environ["NAME"] == "intra":
    raw = data["eeg_proc"].set_eeg_reference(["VREF"])
    evoked = raw.average()
    evoked = mne.set_eeg_reference(evoked, "average")

_ = evoked.plot()

#### Process Evoked

⚙️ <font color=green>**Baseline correction**</font>: To accurately compare EEG channels, they must have the same baseline.

In [25]:
baseline = [0, 0.3]
evoked.apply_baseline(baseline)

_ = evoked.plot()

Applying baseline correction (mode: mean)


⚙️ <font color=green>**Notch Filtering**</font>: A prevalent source of noise in recordings from electrical devices is the 50 Hz electrical interference. To mitigate this type of noise, it is advisable to apply a notch filter, which is a band-stop filter specifically designed to attenuate frequencies around 50 Hz.

In [19]:
evoked.filter(
    l_freq = 63,
    h_freq = 37,
    picks  = "all",
    method = "fir",
)

_ = evoked.plot()

Setting up band-stop filter from 37 - 63 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 37.00
- Lower transition bandwidth: 9.25 Hz (-6 dB cutoff frequency: 32.38 Hz)
- Upper passband edge: 63.00 Hz
- Upper transition bandwidth: 15.75 Hz (-6 dB cutoff frequency: 70.88 Hz)
- Filter length: 91 samples (0.364 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  71 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 161 tasks      | elapsed:    0.0s


## Visualize Topomaps

It is important to check both the EEG time series and the topological maps they generate. A topological map should be smooth; if it is not, the electrodes responsible for the irregularities are likely faulty.

In [12]:
# time points 
times = [0.48]
_ = evoked.plot_topomap(
    times = times,
    outlines = None,
    extrapolate = "local",
)


## 💾 Save processed EEG

In [26]:
# save source space
folder = mm_dir/sub/"eeg"

if folder.exists() == False:
        folder.mkdir(parents=True)

print(f"Save EEG for subject: {sub}")
evoked.save(folder/f"{sub}_BAD_ave.fif", overwrite=True)

Save EEG for subject: sub-pat28
